In [2]:
# =========================================================
# STEP 4A – TOPIC MODELING (UMAP + HDBSCAN + BERTopic)
# =========================================================
import numpy as np
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt

# -------- CONFIG --------
EMB_FILE  = "embeddings_e5.npy"              # embedding 1024 chiều
DATA_FILE = "comments_weaklabel.csv"      # văn bản + weak_label

# -------- LOAD DATA --------
X  = np.load(EMB_FILE)
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
texts = df["text_clean"].fillna("").astype(str).tolist()

print(f"✅ Loaded {len(texts)} comments | Embedding shape: {X.shape}")

# =========================================================
# 1) UMAP – Giảm chiều embedding 1024 → 15
# =========================================================
umap_model = UMAP(
    n_components=15,        #
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# =========================================================
# 2) HDBSCAN – Gom cụm sau khi giảm chiều
# =========================================================
hdbscan_model = HDBSCAN(
    min_cluster_size=500,       # tăng để giảm noise
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# =========================================================
# 3) BERTopic – topic modeling
# =========================================================
topic_model = BERTopic(
    language="multilingual",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    nr_topics=None,     # để thuật toán tự động sinh số topic
    verbose=True
)

print("🔹 Running BERTopic...")
topics, probs = topic_model.fit_transform(texts, X)

# Gán topic ID vào dataframe
df["topic_id"] = topics

# Lấy bảng thống kê topic
topic_info = topic_model.get_topic_info()

# =========================================================
# SAVE RESULTS
# =========================================================
df.to_csv("comments_with_topics.csv", index=False, encoding="utf-8-sig")
topic_info.to_csv("topic_summary.csv", index=False, encoding="utf-8-sig")

print("✅ Saved: comments_with_topics.csv & topic_summary.csv")
print(topic_info.head())

# =========================================================
# VISUALIZATION
# =========================================================
try:
    fig = topic_model.visualize_topics()
    fig.write_html("bertopic_topics.html")
    print("📊 Saved: bertopic_topics.html (interactive)")
except Exception as e:
    print("⚠️ Visualization skipped:", e)

2025-12-06 01:00:46,051 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


✅ Loaded 186744 comments | Embedding shape: (186744, 1024)
🔹 Running BERTopic...


2025-12-06 01:06:50,746 - BERTopic - Dimensionality - Completed ✓
2025-12-06 01:06:50,761 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-06 01:08:03,380 - BERTopic - Cluster - Completed ✓
2025-12-06 01:08:03,411 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-06 01:08:04,940 - BERTopic - Representation - Completed ✓


✅ Saved: comments_with_topics.csv & topic_summary.csv
   Topic  Count                       Name  \
0     -1  81209     -1_không_có_neutral_mà   
1      0  27442             0_pro_15_16_14   
2      1  14132        1_s23_ultra_s24_s25   
3      2   7907         2_bác_không_đâu_mà   
4      3   5626  3_iphone_android_apple_ip   

                                      Representation  \
0  [không, có, neutral, mà, mua, nó, con, cái, má...   
1     [pro, 15, 16, 14, 13, 17, note, mua, hơn, max]   
2  [s23, ultra, s24, s25, s23u, hơn, s24u, dùng, ...   
3   [bác, không, đâu, mà, gì, nói, có, mua, rồi, ơn]   
4  [iphone, android, apple, ip, samsung, người, d...   

                                 Representative_Docs  
0  [điện thoại rẻ tiền mới sài màn hình phẳng ông...  
1  [11 pro max có nên 15 pro max không anh em, vi...  
2  [s23 có không b, cam con thấy vẫn ổn hơn s24 u...  
3  [MENTION_vivantam5191 đâu cần thiết phải thế đ...  
4  [lmao so sao làm iphone bị ép ios cao đ giật k...  
📊 